# TUS-GAN v5.1 (Cell D) - Training & Evaluation Pipeline
**Target Architecture:** Transformer Backbone + Soft Penalty Enforcement  
**Hardware:** NVIDIA A100-80GB (JarvisLabs)  
**Manuscript Reference:** `TUS_GAN_V2.docx` (Table 6 & Section 6.5)


### 1. Verify GPU & Environment

In [1]:
!nvidia-smi
import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))
    print(f"Total VRAM     : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB")

Tue Sep  8 22:06:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.58.03              Driver Version: 595.58.03      CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          On  |   00000000:01:00.0 Off |                    0 |
| N/A   28C    P0             59W /  500W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

### 2. Install Requirements

In [5]:
!pip install -q -r requirements.txt

### 3. Run Pre-Flight Smoke Test
Verifies GPU autograd double-backward pass for WGAN-GP gradient penalty.

In [4]:
!pip install --upgrade pip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 68.9 MB/s  0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 26.1.1
    Uninstalling pip-26.1.1:
      Successfully uninstalled pip-26.1.1


In [ ]:
!python smoke_test.py

### 4. Train TUS-GAN v5.1 (Cell D)
Trains for 250 epochs with batch size 512 using soft penalty enforcement.

In [6]:
# Run single seed (Seed 42)
!python train.py --data tusgan_encode.npz --epochs 250 --batch 512 --seed 42 --checkpoint_dir checkpoints_seed42

🌱 Random seed set to 42

🖥️ Device: cuda
   GPU: NVIDIA A100-SXM4-80GB
✅ [v5.1] Dataset loaded: 445,268 diaries
   Shape: torch.Size([445268, 9, 48, 1]), Cond dim: 82
   Districts: 71, States: 36
Computing real time-slice transition matrices...
Computing real average spell durations...

🚀 Starting Cell D Training (v5.1: Transformer + Soft Penalty) -> 250 Epochs

/root/miniconda3/envs/py3.10/lib/python3.10/site-packages/torch/autograd/graph.py:869: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:335.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
Epoch [   1/250] | W-dist: +0.2746 | Loss D: 3.9369 | Loss G: 1.8766 | Logic: 0.0114
Epoch [   2/250] | W-dist: +0.5937 | Loss D: 0.0591 | Loss G: 1.4170 | Logic: 0.0013
Epoch [   3/250] | W-dist: +1.3351 | Loss D: -1.0898 | Loss G: 4.0815 | Log

### 5. Evaluate Model (Table 6 Metrics)
Computes JSD, Transition F-Norm, Spell-Duration EMD, Child Labor Violation Rates, and Adversarial AUC-ROC.

In [8]:
!python evaluate_ultimate.py \
    --checkpoint checkpoints_seed42/final.pt \
    --data tusgan_encode.npz \
    --n_samples 100000 \
    --output_dir evaluation_100k_results_seed42


🚀 Running TUS-GAN v5.1 Evaluation on cuda...
✅ Loaded real data from tusgan_encode.npz (445,268 diaries)
✅ Loaded Generator from checkpoints_seed42/final.pt (epoch 250, weights: G_state_ema)

Generating 100,000 raw synthetic diaries (soft penalty in training)...
Generating 100,000 delivered synthetic diaries (with inference guarantee)...

 🏆 TUS-GAN v5.1 (CELL D) EVALUATION RESULTS — TABLE 6 METRICS
 📊 Statistical Fidelity:
    - Jensen-Shannon Divergence (JSD)       : 0.004534  (Target: < 0.000100)
 ⏳ Temporal Realism:
    - Transition Matrix F-Norm (4 Slices)   : 0.283401
    - Average Spell-Duration EMD (mins)     : 15.81 mins
    - Employment Spell EMD                  : 27.89 mins
 🛡️ Normative Constraint C1 (Child Labor 3-Point Reporting):
    - 1. Empirical Survey Base Rate         : 873/15,897 (3.87%)
    - 2. Raw Model Output (Soft Penalty)    : 2/14,101 (0.01%)
    - 3. Delivered Output (Inference Mask)  : 0/14,230 (0.00%)
 ⚔️ Adversarial Indistinguishability (Random Forest)

### 6. Display Table 6 Results

In [ ]:
with open("evaluation_results_seed42/table6_cell_d_results.txt", "r") as f:
    print(f.read())

In [9]:
!powershell Compress-Archive -Path "v5.1" -DestinationPath "v5.1.zip"


/bin/bash: line 1: powershell: command not found


In [ ]:
!zip -r v5.1.zip the_folder_you_want_to_zip
